# Explore the warehouse

Look at any table in `data/warehouse.duckdb`. Everything is read-only.

**One rule:** DuckDB allows one writer at a time, and this notebook's read-only
connection blocks writers. Before `pixi run build`, run the last cell (or restart the
kernel). Tables and columns are described in `docs/data-dictionary.md`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                 # the warehouse's staging views read raw files by paths relative to the repo root
sys.path.insert(0, str(ROOT))

import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)

con = duckdb.connect(str(ROOT / "data" / "warehouse.duckdb"), read_only=True)


def q(sql):
    """Run SQL, get a DataFrame:  q("SELECT * FROM marts.mart_mo_il__region_resilience LIMIT 5")"""
    return con.sql(sql).df()

## 1. Every table, with row counts

In [ ]:
tables = q("""
    SELECT table_schema AS schema, table_name AS name, table_type AS type
    FROM information_schema.tables
    WHERE table_schema IN ('seeds', 'staging', 'intermediate', 'marts')
    ORDER BY schema, name
""")
# the national outage view reads ~250M rows of CSV: count it only if you mean to
tables["rows"] = [q(f"SELECT count(*) FROM {s}.{n}").iloc[0, 0] if n != "stg_eaglei__outages" else None
                  for s, n in zip(tables.schema, tables.name)]
tables

## 2. Columns of a table, and a preview

In [ ]:
TABLE = "marts.mart_mo_il__region_6h"
display(q(f"DESCRIBE {TABLE}"))
q(f"SELECT * FROM {TABLE} USING SAMPLE 5 ROWS")

## 3. One region through a storm

In [ ]:
REGION = "Monroe"                        # any region name, e.g. "St. Louis City + County", "Franklin"
START, END = "2025-03-13", "2025-03-18"  # the March 2025 tornado outbreak

ts = q(f"""
    SELECT block_start, peak_pct_out, peak_out, gust_max_ms, cape_max_jkg, precip_sum_mm, storm_reports
    FROM marts.mart_mo_il__region_6h
    WHERE region = '{REGION}' AND block_start BETWEEN '{START}' AND '{END}'
    ORDER BY block_start
""")
fig, (a, b) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)   # two measures, two panels (no dual axis)
a.plot(ts.block_start, ts.peak_pct_out, "o-", color="#eb6834", lw=2)
a.set(ylabel="% customers out", title=f"{REGION}, {START} → {END}")
b.plot(ts.block_start, ts.gust_max_ms, "o-", color="#2a78d6", lw=2)
b.set(ylabel="max gust (m/s)")
for ax in (a, b):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
ts

## 4. The biggest outage blocks

In [ ]:
q("""
    SELECT block_start,
           sum(peak_out)                                AS customers_out,
           count(*) FILTER (WHERE peak_pct_out >= 5)    AS regions_ge_5pct,
           sum(storm_reports)                           AS storm_reports
    FROM marts.mart_mo_il__region_6h
    WHERE peak_out IS NOT NULL
    GROUP BY block_start
    ORDER BY customers_out DESC
    LIMIT 15
""")

## 5. Raw snapshots for a county

Straight from the EAGLE-I files (15-minute snapshots, before any merging). This scans
the national CSVs, so it takes a few seconds.

In [ ]:
FIPS = "29510"          # St. Louis City; try 29189 (St. Louis County) too
q(f"""
    SELECT year(snapshot_time) AS year,
           count(*)                AS snapshots_with_outages,
           max(customers_out)      AS max_out,
           round(avg(customers_out)) AS avg_out
    FROM staging.stg_eaglei__outages
    WHERE fips_code = '{FIPS}'
    GROUP BY year
    ORDER BY year
""")

## 6. Do features separate outage blocks from quiet ones?

Distribution of one feature for blocks with and without a ≥ 5% outage (2024–2025, with
weather). Overlapping boxes = a weak feature on its own.

In [ ]:
FEATURE = "gust_max_ms"   # try: shear_0_6km_max_ms, cape_max_jkg, precip_sum_mm, canopy_mean_pct
d = q(f"""
    SELECT {FEATURE} AS value, peak_pct_out >= 5 AS outage
    FROM marts.mart_mo_il__region_6h
    WHERE has_weather AND peak_pct_out IS NOT NULL
""")
fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot([d.value[~d.outage].dropna(), d.value[d.outage].dropna()], vert=False, showfliers=False,
           tick_labels=["no outage", ">= 5% out"])
ax.set(xlabel=FEATURE, title=f"{FEATURE}: quiet blocks vs outage blocks")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
d.groupby("outage").value.describe().round(2)

## 7. Map a region metric

In [ ]:
METRIC = "storm_hit_rate"   # any column of marts.mart_mo_il__region_resilience
shapes = gpd.read_parquet(ROOT / "data" / "raw" / "geo" / "regions_mo_il.parquet")
metric = q(f"SELECT region_fips, {METRIC} FROM marts.mart_mo_il__region_resilience")
ax = shapes.merge(metric, on="region_fips").plot(column=METRIC, cmap="Blues", legend=True, figsize=(7, 8),
                                                 edgecolor="white", linewidth=0.3)
ax.set_axis_off()
ax.set_title(METRIC)
plt.show()

## Your own query

In [ ]:
q("""
    SELECT state, count(*) AS regions, round(avg(canopy_mean_pct), 1) AS avg_canopy_pct
    FROM intermediate.int_mo_il__regions
    GROUP BY state
""")

## Done? Release the database (needed before `pixi run build`)

In [ ]:
con.close()